## BOPC comparison

**Goal:** See how the data DPD sent to BOPC in weekly reports matches up with data they reported in the FOIA response.

In [1]:
#import data
import pandas as pd
main = pd.read_excel('data/shotspotter_2021_2026_main.xlsx')
arrests = pd.read_excel('data/shotspotter_2021_2026_arrests.xlsx')
firearms = pd.read_excel('data/shotspotter_2021_2026_firearms.xlsx')
bopc_data = pd.read_csv('data/weekly_bopc_shotspotter.csv')

In [2]:
#revise column names
main.columns = ['id', 'call_time', 'rms', 'nature', 'statbeat', 'pct', 'source', 'street_add']
arrests.columns = ['rms', 'arrests']
firearms.columns = ['rms', 'firearms_recovered']

#merge data sets on rms column
foia_data = pd.merge(main, arrests, on='rms', how='left')
foia_data = pd.merge(foia_data, firearms, on='rms', how='left')

#replace null values with 0
foia_data['arrests'] = foia_data['arrests'].fillna(0)
foia_data['firearms_recovered'] = foia_data['firearms_recovered'].fillna(0)
#convert call_time to datetime
foia_data['call_time'] = pd.to_datetime(foia_data['call_time'])
#extract year and month from call_time
foia_data['year'] = foia_data['call_time'].dt.year
foia_data['month'] = foia_data['call_time'].dt.month

foia_data.head()

,id,call_time,rms,nature,statbeat,pct,source,street_add,arrests,firearms_recovered,year,month
0,202113500265,2021-05-15 00:57:18,2.105150e+09,SHOT SPOTTER,807.0,8.0,SELF,18669 FAUST AVE,0.0,0.0,2021,5
1,202113500306,2021-05-15 01:09:03,NaN,SHOT SPOTTER,906.0,9.0,PHONE,12540 HAYES ST,0.0,0.0,2021,5
2,202113500315,2021-05-15 01:11:37,2.105150e+09,SHOT SPOTTER,807.0,8.0,SELF,19841 CURTIS ST,0.0,0.0,2021,5
3,202113500508,2021-05-15 02:10:43,2.105150e+09,SHOT SPOTTER,807.0,8.0,SELF,18997 ANNCHESTER RD,0.0,0.0,2021,5
4,202113500651,2021-05-15 02:53:02,NaN,SHOT SPOTTER,902.0,9.0,PHONE,13801 ROSSINI DR,0.0,0.0,2021,5


In [12]:
foia_data.columns

Index(['id', 'call_time', 'rms', 'nature', 'statbeat', 'pct', 'source',
       'street_add', 'arrests', 'firearms_recovered', 'year', 'month'],
      dtype='str')

In [13]:
bopc_data.columns

Index(['page', 'precinct', 'date', 'incidents_last_week',
       'shots_fired_last_week', 'guns_recovered_last_week'],
      dtype='str')

**Overall comparison**

In [5]:
min = bopc_data['date'].min()
max = bopc_data['date'].max()

#foia data between dates
foia_bw_dates = foia_data[(foia_data['call_time'] <= max) & (foia_data['call_time'] >= min)]

#bopc data between dates
bopc_bw_dates = bopc_data[(bopc_data['date'] <= max) & (bopc_data['date'] >= min)]

print("FOIA:", len(foia_bw_dates))
print("BOPC:", len(bopc_bw_dates))

FOIA: 47879
BOPC: 682


In [7]:
#add up incidents, guns recovered from both data sets

#bopc total incidents
print(bopc_bw_dates['incidents_last_week'].sum())

#bopc total guns recovered
print(bopc_bw_dates['guns_recovered_last_week'].sum())

#foia total guns recovered
print(foia_bw_dates['firearms_recovered'].sum())

19922
772.0
1805.0


**Hypothetically, the number of incidents, arrests and guns seized in the week leading up to each report date (in the FOIA data) should match up with the numbers in each report.**

In [16]:
#Pick a random week and precinct (not in 2021 or 2024)
bopc_data[(bopc_data['date'] == '2022-01-24') & (bopc_data['precinct'] == 9)]

,page,precinct,date,incidents_last_week,shots_fired_last_week,guns_recovered_last_week
1,2,9,2022-01-24,30,114,1.0


In [17]:
#what are the corresponding numbers from the foia data set?
selected_week = foia_data[(foia_data['call_time'] >= '2022-01-17') & (foia_data['call_time'] <= '2022-01-23') & (foia_data['pct'] == 9)]

print(len(selected_week))
print(selected_week['arrests'].sum())
print(selected_week['firearms_recovered'].sum())

40
0.0
1.0


Those numbers don't seem to match up. Let's look across a larger sample.

In [18]:
#get all report dates
report_dates = bopc_data['date'].unique()
len(report_dates)

120

In [20]:
# for each report date, count incidents and sum firearms recovered in the 7 days prior (ending day before report)
results = []
for d in pd.to_datetime(report_dates):
    start = d - pd.Timedelta(days=7)
    end = d - pd.Timedelta(days=1)
    mask = (foia_data['call_time'] >= start) & (foia_data['call_time'] <= end)
    subset = foia_data.loc[mask]
    results.append({
        'date': pd.to_datetime(d),
        'incidents_last_week': subset.shape[0],
        'firearms_recovered_last_week': subset['firearms_recovered'].sum()
    })

weekly_counts_df = pd.DataFrame(results).sort_values('date').reset_index(drop=True)
weekly_counts_df

,date,incidents_last_week,firearms_recovered_last_week
0,2022-01-24,50,2.0
1,2022-01-31,45,1.0
2,2022-02-07,55,0.0
3,2022-02-14,36,6.0
4,2022-02-21,32,1.0
...,...,...,...
115,2025-12-08,140,4.0
116,2025-12-15,119,2.0
117,2025-12-22,97,4.0
118,2025-12-29,165,11.0


In [23]:
#now, let's group all precinct reports from one report date together
bopc_weekly_df = bopc_data[['date', 'incidents_last_week', 'guns_recovered_last_week']].groupby('date').sum().reset_index()
bopc_weekly_df.head()

,date,incidents_last_week,guns_recovered_last_week
0,2022-01-24,37,2.0
1,2022-01-31,25,1.0
2,2022-02-07,25,0.0
3,2022-02-14,8,1.0
4,2022-02-21,34,2.0


In [26]:
#now, let's compare by merging on date
weekly_counts_df.columns = ['date', 'incidents_last_week', 'guns_recovered_last_week']
bopc_weekly_df['date'] = pd.to_datetime(bopc_weekly_df['date']) 
merged_weekly_counts = pd.merge(weekly_counts_df, bopc_weekly_df, on='date', suffixes=['_foia', '_bopc'])
merged_weekly_counts.head()

,date,incidents_last_week_foia,guns_recovered_last_week_foia,incidents_last_week_bopc,guns_recovered_last_week_bopc
0,2022-01-24,50,2.0,37,2.0
1,2022-01-31,45,1.0,25,1.0
2,2022-02-07,55,0.0,25,0.0
3,2022-02-14,36,6.0,8,1.0
4,2022-02-21,32,1.0,34,2.0


In [27]:
# proportion of weeks where FOIA and BOPC incident counts match
eq_mask = merged_weekly_counts['incidents_last_week_foia'] == merged_weekly_counts['incidents_last_week_bopc']
num_equal = eq_mask.sum()
total = merged_weekly_counts.shape[0]
pct_equal = num_equal / total * 100

print(f"{num_equal} out of {total} weeks equal ({pct_equal:.1f}%)")

2 out of 120 weeks equal (1.7%)


In [28]:
# proportion of weeks where FOIA and BOPC guns recovered counts match
eq_mask = merged_weekly_counts['guns_recovered_last_week_foia'] == merged_weekly_counts['guns_recovered_last_week_bopc']
num_equal = eq_mask.sum()
total = merged_weekly_counts.shape[0]
pct_equal = num_equal / total * 100

print(f"{num_equal} out of {total} weeks equal ({pct_equal:.1f}%)")

15 out of 120 weeks equal (12.5%)


The gun recovery numbers only matched 12.5% of the time and the incident numbers only matched 1.7% of the time. 

In [30]:
#How far apart are the sums?
print("BOPC data")
print("Total incidents", bopc_weekly_df['incidents_last_week'].sum())
print("Guns recovered", bopc_weekly_df['guns_recovered_last_week'].sum())
print()

print("FOIA data (limited to overlap period)")
print("Total incidents", weekly_counts_df['incidents_last_week'].sum())
print("Guns recovered", weekly_counts_df['guns_recovered_last_week'].sum())
print()

BOPC data
Total incidents 19922
Guns recovered 772.0

FOIA data (limited to overlap period)
Total incidents 17903
Guns recovered 832.0



**So, for the weeks that DPD reported incident and gun recovery totals to BOPC, the FOIA data showed 11% fewer incidents and 8% more firearms recovered.**

Let's look a little deeper at that example week:

In [43]:
len(selected_week[selected_week['source'] == 'PHONE'])

30

That could be our issue -- I don't know why "source" is sometimes SELF and sometimes PHONE, but when I limit the FOIA data to just the PHONE source, the incident count matches up with the BOPC report. Let's try that across the whole data set.

In [64]:
# for each report date, count incidents and sum firearms recovered in the 7 days prior (ending day before report)
results = []

foia_data_single_source = foia_data[foia_data['source'].isin(['SPOT', 'SELF'])]

for d in pd.to_datetime(report_dates):
    start = d - pd.Timedelta(days=7)
    end = d - pd.Timedelta(days=1)
    mask = (foia_data_single_source['call_time'] >= start) & (foia_data_single_source['call_time'] <= end)
    subset = foia_data_single_source.loc[mask]
    results.append({
        'date': pd.to_datetime(d),
        'incidents_last_week': subset.shape[0],
        'firearms_recovered_last_week': subset['firearms_recovered'].sum()
    })

weekly_counts_single_source = pd.DataFrame(results).sort_values('date').reset_index(drop=True)
weekly_counts_single_source

,date,incidents_last_week,firearms_recovered_last_week
0,2022-01-24,13,1.0
1,2022-01-31,5,0.0
2,2022-02-07,11,0.0
3,2022-02-14,1,0.0
4,2022-02-21,3,1.0
...,...,...,...
115,2025-12-08,140,4.0
116,2025-12-15,119,2.0
117,2025-12-22,97,4.0
118,2025-12-29,163,11.0


In [65]:
#now, let's compare by merging on date
weekly_counts_single_source.columns = ['date', 'incidents_last_week', 'guns_recovered_last_week']
bopc_weekly_df['date'] = pd.to_datetime(bopc_weekly_df['date']) 
merged_weekly_counts = pd.merge(weekly_counts_single_source, bopc_weekly_df, on='date', suffixes=['_foia', '_bopc'])
merged_weekly_counts.head()

,date,incidents_last_week_foia,guns_recovered_last_week_foia,incidents_last_week_bopc,guns_recovered_last_week_bopc
0,2022-01-24,13,1.0,37,2.0
1,2022-01-31,5,0.0,25,1.0
2,2022-02-07,11,0.0,25,0.0
3,2022-02-14,1,0.0,8,1.0
4,2022-02-21,3,1.0,34,2.0


In [66]:
#filter to 2023 or more recent
merged_weekly_counts_since_23 = merged_weekly_counts[merged_weekly_counts['date'] > "2023-06-01"]

In [67]:
# proportion of weeks where FOIA and BOPC incident counts match
eq_mask = merged_weekly_counts_since_23['incidents_last_week_foia'] == merged_weekly_counts_since_23['incidents_last_week_bopc']
num_equal = eq_mask.sum()
total = merged_weekly_counts_since_23.shape[0]
pct_equal = num_equal / total * 100

print(f"{num_equal} out of {total} weeks equal ({pct_equal:.1f}%)")

0 out of 56 weeks equal (0.0%)


In [68]:
# proportion of weeks where FOIA and BOPC guns recovered counts match
eq_mask = merged_weekly_counts_since_23['guns_recovered_last_week_foia'] == merged_weekly_counts_since_23['guns_recovered_last_week_bopc']
num_equal = eq_mask.sum()
total = merged_weekly_counts_since_23.shape[0]
pct_equal = num_equal / total * 100

print(f"{num_equal} out of {total} weeks equal ({pct_equal:.1f}%)")

0 out of 56 weeks equal (0.0%)


Incident numbers don't get any better, but the gun recovery values do. I want to know more about the source column.

In [50]:
foia_data['source'].value_counts()

source
SPOT     43195
PHONE     8321
SELF      2469
RPTO        33
E911         6
W911         5
RADIO        3
Name: count, dtype: int64

In [59]:
#breakdown by year
foia_data[['id', 'year', 'source']].groupby(['year', 'source']).count().reset_index().pivot(
    index='year', columns = 'source', values='id').reset_index()

source,year,E911,PHONE,RADIO,RPTO,SELF,SPOT,W911
0,2021,4.0,2694.0,1.0,3.0,416.0,NaN,NaN
1,2022,NaN,3346.0,2.0,2.0,614.0,NaN,1.0
2,2023,1.0,2106.0,NaN,11.0,771.0,15276.0,3.0
3,2024,NaN,70.0,NaN,6.0,218.0,14475.0,1.0
4,2025,1.0,98.0,NaN,8.0,314.0,10409.0,NaN
5,2026,NaN,7.0,NaN,3.0,136.0,3035.0,NaN


It looks like most of the alerts were classified as "PHONE" in 2021 and 2022, but most were "SPOT" in 2023, 2024, 2025 and 2026. I'd like to know more about why that is.